In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

os.environ["HF_HUB_TIMEOUT"] = "120"

os.environ["HF_HUB_OFFLINE"] = "0"

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 定义模型 ID
model_id = "Qwen/Qwen2.5-7B-Instruct"

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 定义4-bit量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/home/ricaedo/下载/yes/envs/AutoGluon/lib/python3.9/site-packages/accelerate/utils/modeling.py:1582: UserWarning: Current model requires 32.0 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), 

In [4]:
def chat(user_message, system_message="你是《黑神话：悟空》领域助手，回答准确、简明。"):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]
    
    # 应用对话模板
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # 模型生成
    generated_ids = model.generate(
        input_ids=model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=256
    )
    
    # 解码时跳过 prompt 部分
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response

In [5]:
question_1 = "我该怎么成为天命人？"
answer_1 = chat(question_1)
print(f"问题: {question_1}\n回答:\n{answer_1}")

问题: 我该怎么成为天命人？
回答:
在《黑神话：悟空》的设定中，成为天命人是通过一系列特殊的修炼和考验获得的称号。具体步骤如下：

1. 达到一定等级：角色需要达到一定的等级。
2. 完成特定任务或挑战：完成游戏中的特定任务或挑战，以证明你的实力和潜力。
3. 接受试炼：接受来自神秘力量或前辈的试炼，证明自己是否具备成为天命人的资格。

请注意，以上信息基于目前公开的游戏设定，实际游戏内容可能会有所不同。


In [6]:
question_2 = "如何获得并合成出云棍？"
answer_2 = chat(question_2)
print(f"问题: {question_2}\n回答:\n{answer_2}")

问题: 如何获得并合成出云棍？
回答:
在《黑神话：悟空》中，获得并合成出云棍的步骤如下：

1. **收集材料**：你需要收集特定的材料来制作出云棍。这些材料包括但不限于特定的草药、矿石和符文。

2. **前往特定地点**：游戏中的某些区域可能需要你去特定的地方才能找到制作所需的材料或进行合成。

3. **合成过程**：将收集到的材料带到指定的地点或者NPC处进行合成。通常需要正确排列材料顺序，并满足一定的条件才能成功合成出云棍。

4. **完成任务**：有时还需要完成一些前置任务或挑战，以解锁出云棍的合成权限。

请注意，具体细节可能会根据游戏更新而有所变化，请参考最新的游戏指南或官方说明。
